# 02b Prep: Creating Training data for Semantic segmentation
Includes:  
- Tiling & Chipping  
- Batch & Pairing  
- Quality Check

In [3]:
# load libraries
from pathlib import Path
import geoai
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import geopandas as gpd
import rasterio
from rasterio.windows import Window

In [6]:
# raster and vector mask paths
ortho_path = "../data/raw/raster/vexcel/luzern-2025_epsg2056.tif"
mask_path = "../data/processed/combined_mask_NDVI01.tif"

In [7]:
# Verify raster alignment
with rasterio.open(ortho_path) as ortho:
    print("ORTHO")
    print(ortho.shape)
    print(ortho.crs)
    print(ortho.bounds)

with rasterio.open(mask_path) as mask:
    print("\nMASK")
    print(mask.shape)
    print(mask.crs)
    print(mask.bounds)

ORTHO
(13334, 13334)
EPSG:2056
BoundingBox(left=2666000.0, bottom=1209999.95, right=2667000.05, top=1211000.0)

MASK
(13334, 13334)
EPSG:2056
BoundingBox(left=2666000.0, bottom=1209999.95, right=2667000.05, top=1211000.0)


#### Subsetting AoI for smaller area
For the first run, i want to reduce the amount of data and therefore only look at a minimal amount of my original area of interest.

In [8]:
# Output files
ortho_subset = "../data/processed/ortho_subset_center01.tif"
mask_subset = "../data/processed/mask_subset_center01.tif"

crop_size = 6667 # ca. 500m at a 7.5cm resolution

with rasterio.open(ortho_path) as src:

    # center vertically, left quarter horizontally
    col_center = src.width // 2
    row_center = src.height // 2

    window = Window(
        col_center - crop_size // 2,
        row_center - crop_size // 2,
        crop_size,
        crop_size
    )

    data = src.read(window=window)

    profile = src.profile.copy()
    profile.update(
        width=crop_size,
        height=crop_size,
        transform=rasterio.windows.transform(window, src.transform)
    )

    with rasterio.open(ortho_subset, "w", **profile) as dst:
        dst.write(data)


In [ ]:
# Apply identical crop to mask
with rasterio.open(mask_path) as src:

    data = src.read(window=window)

    profile = src.profile.copy()
    profile.update(
        width=crop_size,
        height=crop_size,
        transform=rasterio.windows.transform(window, src.transform)
    )

    with rasterio.open(mask_subset, "w", **profile) as dst:
        dst.write(data)


#### Tiling and Chipping

According to this Docstring for the *geoai.export_geotiff_tiles function* it expects EITHER vector OR raster data for the in_class_data  
This is perfect as we already have rasterized my vector mask (also did the raster calculations, would not make much sense to already vectorize back)  

In the next step we need to define the parameters for that specific GeoAI function in order to create image tiles.

In [6]:
# Selecting tile size and stride
### starting with a smaller/moderarte overlap of 25% before scaling up
tile_size = 512  # Size of each tile (height, width)
stride_size = 384  # Stride for sliding window (height, width)

# define input path
ortho_subset = "../data/processed/ortho_subset_center01.tif"
mask_subset = "../data/processed/mask_subset_center01.tif"

# Define output path
output_dir = Path("../data/training/tiles")

In [8]:
# Generating tiles from the orthophoto and mask
## based on workflow from gitbook
if not hasattr(cm, "get_cmap"):
    cm.get_cmap = plt.get_cmap

tiles = geoai.export_geotiff_tiles(
    in_raster=ortho_subset,
    out_folder=output_dir,
    in_class_data=mask_subset,
    tile_size=tile_size,
    stride=stride_size,
    skip_empty_tiles=True, # added this, to skip tiles that do not contain any buildings
    buffer_radius=0,
    create_overview=True,
    quiet=True,
)

: 

In [ ]:
from pathlib import Path

for p in Path("data/training/tiles").rglob("*"):
    print(p)

In [ ]:
# inspect image-mask pairs in overview.png to confirm that tiles were generated were we expected
geoai.view_image("output/overview.png", figsize=(18, 10))

In [ ]:
fig = geoai.display_training_tiles(
    output_dir="output",
    num_tiles=8,
    figsize=(18, 10),
)

# verify whether all roofs are fully visible
## whether masks align with buildings
### whether enough positive samples exist
#### whether no obvious shift between imagery and labels 

#### Batch & Pairing

#### Quality & splitting